# Interactive Reranking Explorer

Compare **Hybrid Fusion**, **Distilled Pairwise**, and **Shallow ColBERT**
reranking strategies side-by-side on your own queries and documents.

All models run **CPU-only** — no GPU or API calls needed.

In [1]:
import json
import time
from pathlib import Path

import numpy as np

## Load Data

Load training pairs and preferences from `data/raw/` if available, otherwise use built-in sample data.

In [2]:
pairs_path = Path("data/raw/pairs.jsonl")
preferences_path = Path("data/raw/preferences.jsonl")

pairs_data = []
preferences_data = []

if pairs_path.exists():
    with open(pairs_path) as f:
        pairs_data = [json.loads(line) for line in f if line.strip()]

if preferences_path.exists():
    with open(preferences_path) as f:
        preferences_data = [json.loads(line) for line in f if line.strip()]

# Fall back to hardcoded sample data if no files found
if not pairs_data:
    pairs_data = [
        {"query": "python dataclass default factory", "doc": "In Python, a dataclass field can use default_factory to generate mutable default values like lists or dictionaries.", "score": 1.0, "rationale": "direct explanation"},
        {"query": "python dataclass default factory", "doc": "Dataclasses in Python 3.7+ provide a decorator that automatically generates __init__, __repr__, and __eq__ methods.", "score": 0.6, "rationale": "related but indirect"},
        {"query": "python dataclass default factory", "doc": "The default_factory parameter in field() is called when a default value is needed for a new instance.", "score": 0.9, "rationale": "close match"},
        {"query": "python dataclass default factory", "doc": "Use dataclasses.field(default_factory=list) to create a new list for each instance rather than sharing one.", "score": 0.8, "rationale": "specific example"},
        {"query": "python dataclass default factory", "doc": "FastAPI relies heavily on Python dataclasses and Pydantic models for request validation.", "score": 0.3, "rationale": "tangential"},
        {"query": "machine learning model deployment", "doc": "ML model deployment can use FastAPI to serve predictions via REST endpoints with automatic OpenAPI docs.", "score": 0.9, "rationale": "good example"},
        {"query": "machine learning model deployment", "doc": "XGBoost and LightGBM models can be serialized and served through ONNX Runtime for production inference.", "score": 0.7, "rationale": "useful detail"},
        {"query": "machine learning model deployment", "doc": "Containerizing ML models with Docker ensures reproducible deployment across environments.", "score": 0.5, "rationale": "related context"},
        {"query": "fastapi async request validation", "doc": "FastAPI relies heavily on Python dataclasses and Pydantic models for request validation.", "score": 1.0, "rationale": "exact match"},
        {"query": "fastapi async request validation", "doc": "ML model deployment can use FastAPI to serve predictions via REST endpoints with automatic OpenAPI docs.", "score": 0.8, "rationale": "good example"},
    ]

if not preferences_data:
    preferences_data = [
        {"query": "python dataclass default factory", "doc_a": pairs_data[0]["doc"], "doc_b": pairs_data[4]["doc"], "preferred": "A", "confidence": 1.0},
        {"query": "python dataclass default factory", "doc_a": pairs_data[2]["doc"], "doc_b": pairs_data[1]["doc"], "preferred": "A", "confidence": 0.9},
        {"query": "python dataclass default factory", "doc_a": pairs_data[3]["doc"], "doc_b": pairs_data[1]["doc"], "preferred": "A", "confidence": 0.8},
        {"query": "machine learning model deployment", "doc_a": pairs_data[5]["doc"], "doc_b": pairs_data[6]["doc"], "preferred": "A", "confidence": 0.8},
        {"query": "machine learning model deployment", "doc_a": pairs_data[5]["doc"], "doc_b": pairs_data[7]["doc"], "preferred": "A", "confidence": 0.9},
        {"query": "fastapi async request validation", "doc_a": pairs_data[8]["doc"], "doc_b": pairs_data[9]["doc"], "preferred": "A", "confidence": 0.7},
    ]

unique_queries = sorted(set(p["query"] for p in pairs_data))
unique_docs = sorted(set(p["doc"] for p in pairs_data))

print(f"Loaded {len(pairs_data)} pairs, {len(preferences_data)} preferences")
print(f"  {len(unique_queries)} unique queries, {len(unique_docs)} unique docs")

Loaded 10 pairs, 6 preferences
  3 unique queries, 8 unique docs


## Initialize Models

In [3]:
from reranker.embedder import Embedder
from reranker.strategies.distilled import DistilledPairwiseRanker
from reranker.strategies.hybrid import HybridFusionReranker
from reranker.strategies.late_interaction import StaticColBERTReranker

embedder = Embedder()
backend_info = embedder.describe()
dim = embedder.dimension
print(f"Embedder: {backend_info['backend']} | Model: {backend_info['model_name']} | Dim: {dim}")

/Users/minghao/Desktop/personal/shallow_cross_encoders/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Embedder: model2vec | Model: minishlab/potion-base-32M | Dim: 512


## Train Strategies

In [4]:
def train_hybrid(pairs, emb):
    reranker = HybridFusionReranker(embedder=emb)
    queries = [p["query"] for p in pairs]
    docs = [p["doc"] for p in pairs]
    scores = [float(p["score"]) for p in pairs]
    reranker.fit_pointwise(queries, docs, scores, use_regression=True)
    return reranker

def train_distilled(prefs, emb):
    reranker = DistilledPairwiseRanker(embedder=emb)
    queries = [p["query"] for p in prefs]
    doc_as = [p["doc_a"] for p in prefs]
    doc_bs = [p["doc_b"] for p in prefs]
    labels = [1 if p["preferred"] == "A" else 0 for p in prefs]
    reranker.fit(queries, doc_as, doc_bs, labels)
    return reranker

def train_colbert(pairs, emb):
    reranker = StaticColBERTReranker(embedder=emb)
    all_docs = list(set(p["doc"] for p in pairs))
    reranker.fit(all_docs)
    return reranker

print("Training Hybrid Fusion...")
hybrid_reranker = train_hybrid(pairs_data, embedder)
print(f"  Fitted: {hybrid_reranker.is_fitted} | Features: {len(hybrid_reranker.feature_names_)}")

print("Training Distilled Pairwise...")
distilled_reranker = train_distilled(preferences_data, embedder)
print(f"  Fitted: {distilled_reranker.is_fitted}")

print("Initializing Shallow ColBERT...")
colbert_reranker = train_colbert(pairs_data, embedder)

print("\nAll strategies ready.")

Training Hybrid Fusion...
  Fitted: True | Features: 9
Training Distilled Pairwise...
  Fitted: True
Initializing Shallow ColBERT...

All strategies ready.


## Compare Reranking Results

Pick a query and see how each strategy ranks the documents.

In [5]:
test_query = unique_queries[0]
test_docs = unique_docs

print(f"Query: {test_query}")
print(f"Documents: {len(test_docs)}")
print()

def run_rerank(reranker, query, docs, label):
    start = time.perf_counter()
    results = reranker.rerank(query, docs)
    elapsed_ms = (time.perf_counter() - start) * 1000
    return {"label": label, "latency_ms": elapsed_ms, "results": results}

all_results = []
if hybrid_reranker:
    all_results.append(run_rerank(hybrid_reranker, test_query, test_docs, "Hybrid Fusion"))
if distilled_reranker:
    all_results.append(run_rerank(distilled_reranker, test_query, test_docs, "Distilled Pairwise"))
if colbert_reranker:
    all_results.append(run_rerank(colbert_reranker, test_query, test_docs, "Shallow ColBERT"))

for res_item in all_results:
    print(f"\n## {res_item['label']}")
    print(f"**Latency:** {res_item['latency_ms']:.2f}ms")
    print()
    print(f"{'Rank':<6} {'Score':<8} Document")
    print("-" * 60)
    for r in res_item["results"]:
        doc_preview = r.doc[:80] + ("..." if len(r.doc) > 80 else "")
        print(f"{r.rank:<6} {r.score:<8.4f} {doc_preview}")

Query: fastapi async request validation
Documents: 8


## Hybrid Fusion
**Latency:** 1.46ms

Rank   Score    Document
------------------------------------------------------------
1      1.0628   FastAPI relies heavily on Python dataclasses and Pydantic models for request val...
2      0.6038   ML model deployment can use FastAPI to serve predictions via REST endpoints with...
3      0.3835   Dataclasses in Python 3.7+ provide a decorator that automatically generates __in...
4      0.3769   The default_factory parameter in field() is called when a default value is neede...
5      0.3738   In Python, a dataclass field can use default_factory to generate mutable default...
6      0.3641   Use dataclasses.field(default_factory=list) to create a new list for each instan...
7      0.3637   XGBoost and LightGBM models can be serialized and served through ONNX Runtime fo...
8      0.3465   Containerizing ML models with Docker ensures reproducible deployment across envi...

## Distilled Pairwis

## Latency Comparison

In [6]:
print(f"{'Strategy':<22} {'Latency (ms)':<15}")
print("-" * 40)
for r in sorted(all_results, key=lambda x: x["latency_ms"]):
    print(f"{r['label']:<22} {r['latency_ms']:<15.2f}")

if len(all_results) >= 2:
    fastest = min(r["latency_ms"] for r in all_results)
    slowest = max(r["latency_ms"] for r in all_results)
    print(f"\nFastest / Slowest ratio: {slowest / fastest:.1f}x")

Strategy               Latency (ms)   
----------------------------------------
Shallow ColBERT        0.37           
Distilled Pairwise     0.40           
Hybrid Fusion          1.46           

Fastest / Slowest ratio: 3.9x
